# 27 — Multimodal Learning: Combining Ultrasound Images and Clinical Features

In the previous notebook, we studied robustness, domain shift, scanner/site differences, and harmonization.

Now we will move to another major real-world setting:

> **A patient often has more information than the ultrasound image alone.**

A clinical prediction system may have access to:

- Ultrasound images
- Age
- Laboratory measurements
- Symptoms
- Prior history
- Categorical clinical variables
- Scanner/site metadata

A multimodal model combines information from more than one modality.

For this notebook:

$$
\boxed{
\text{Ultrasound Image}
+
\text{Clinical Features}
\rightarrow
\text{Prediction}
}
$$

## In this notebook, we will study:

1. Why combine imaging and tabular data?
2. Clinical feature preprocessing
3. Missing-value handling
4. Numerical and categorical variables
5. Image encoder
6. Tabular encoder
7. Feature fusion
8. Early vs late fusion
9. Concatenation-based multimodal networks
10. Training multimodal models
11. Preventing leakage from clinical variables
12. Modality ablation studies
13. Comparing image-only vs tabular-only vs fused models
14. Patient-level multimodal evaluation
15. Handling missing modalities
16. Preparing a research-quality multimodal ultrasound pipeline

## Main Goal

We want to understand the complete multimodal path:

$$
\boxed{
Image
\rightarrow
Image\ Encoder
\rightarrow
Image\ Features
}
$$

and:

$$
\boxed{
Clinical\ Variables
\rightarrow
Tabular\ Encoder
\rightarrow
Clinical\ Features
}
$$

then:

$$
\boxed{
Image\ Features
+
Clinical\ Features
\rightarrow
Fusion
\rightarrow
Classifier
}
$$

The central research principle is:

> **Every modality must be available at the intended prediction time, and every preprocessing step must be fitted using training data only.**


In [ ]:
import copy
import json
import math
import random
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import (
    Dataset,
    DataLoader
)

print("PyTorch:", torch.__version__)


# 1. What Is Multimodal Learning?

A modality is one type/source of information.

Examples:

$$
\begin{array}{|c|c|}
\hline
\textbf{Modality} & \textbf{Examples} \\
\hline
Image & Ultrasound,\ CT,\ MRI \\
\hline
Tabular & Age,\ BMI,\ labs \\
\hline
Text & Clinical\ notes \\
\hline
Waveform & ECG,\ PPG \\
\hline
Genomics & Gene\ expression \\
\hline
\end{array}
$$

Multimodal learning tries to combine complementary information.


# 2. Why Combine Imaging and Clinical Features?

An ultrasound image may contain:

- Anatomy
- Shape
- Texture
- Echogenicity
- Acoustic artifacts

Clinical variables may contain:

- Age-related risk
- Symptoms
- Laboratory information
- Prior history
- Physiological context

If the modalities contain complementary signal:

$$
\boxed{
Information(Image,Clinical)
>
Information(Image)
}
$$

a fused model may outperform either modality alone.


# 3. Multimodal Does Not Automatically Mean Better

Adding more variables can hurt because of:

- Noise
- Missingness
- Leakage
- Small sample size
- Redundant features
- Different scaling
- Overfitting
- Deployment mismatch

So multimodal learning must be validated with:

> **Modality ablation studies**


# 4. The Three Core Baselines

A strong multimodal project should compare:

$$
\boxed{
Image\ Only
}
$$

$$
\boxed{
Tabular\ Only
}
$$

$$
\boxed{
Image+Tabular
}
$$

Without those baselines, you cannot tell which modality contributes useful information.


# 5. Prediction-Time Availability

Before using a feature, ask:

> **Would this feature truly exist when the model is supposed to make its prediction?**

Dangerous examples:

- Final pathology result
- Post-treatment lab result
- Radiologist final diagnosis
- Surgery outcome
- Future medication
- Test ordered because clinicians already knew the diagnosis

These create **temporal or clinical leakage**.


# 6. Leakage Example

Suppose the task is:

> Predict malignancy before biopsy.

Using:

```text
biopsy_result
```

as a tabular feature would make performance meaningless.

The model is being given the answer.

A good multimodal pipeline starts by defining:

$$
\boxed{
Prediction\ Time
}
$$

and only uses information available before that moment.


# 7. Synthetic Multimodal Dataset

To keep this notebook runnable without external files, we will create synthetic patient data containing:

- Multiple ultrasound images per patient
- Three classes
- Numerical clinical variables
- Categorical clinical variables
- Missing values
- Site/device metadata

The goal is to learn the multimodal pipeline, not to simulate real medicine perfectly.


In [ ]:
def make_multimodal_image(
    class_index,
    image_size,
    generator
):
    image = torch.zeros(
        1,
        image_size,
        image_size
    )

    center = image_size // 2

    shift_y = int(
        torch.randint(
            -4,
            5,
            (1,),
            generator=generator
        ).item()
    )

    shift_x = int(
        torch.randint(
            -4,
            5,
            (1,),
            generator=generator
        ).item()
    )

    cy = center + shift_y
    cx = center + shift_x

    if class_index == 0:
        image[
            :,
            6:image_size - 6,
            max(
                0,
                cx - 2
            ):
            min(
                image_size,
                cx + 3
            )
        ] = 0.9

    elif class_index == 1:
        image[
            :,
            max(
                0,
                cy - 2
            ):
            min(
                image_size,
                cy + 3
            ),
            6:image_size - 6
        ] = 0.9

    elif class_index == 2:
        image[
            :,
            max(
                0,
                cy - 7
            ):
            min(
                image_size,
                cy + 8
            ),
            max(
                0,
                cx - 7
            ):
            min(
                image_size,
                cx + 8
            )
        ] = 0.9

    noise = torch.randn(
        image.shape,
        generator=generator
    ) * 0.12

    return (
        image
        + noise
    ).clamp(
        0.0,
        1.0
    )


# 8. Clinical Feature Design

We will create numerical features:

$$
\begin{array}{|c|c|}
\hline
age & \text{Years} \\
\hline
bmi & \text{Continuous value} \\
\hline
biomarker & \text{Continuous laboratory-style value} \\
\hline
\end{array}
$$

and categorical features:

$$
\begin{array}{|c|c|}
\hline
sex & F/M \\
\hline
symptom\_group & A/B/C \\
\hline
\end{array}
$$


In [ ]:
def create_multimodal_patient_data(
    num_patients=120,
    images_per_patient=2,
    image_size=48
):
    images = []
    records = []

    sites = [
        "Site_A",
        "Site_B",
        "Site_C"
    ]

    devices = [
        "Device_A",
        "Device_B",
        "Device_C"
    ]

    symptom_options = [
        "A",
        "B",
        "C"
    ]

    for patient_index in range(
        num_patients
    ):
        patient_id = (
            f"P{patient_index:03d}"
        )

        class_index = (
            patient_index
            % 3
        )

        generator = (
            torch.Generator()
            .manual_seed(
                10000
                + patient_index
            )
        )

        age = (
            35.0
            + 10.0
            * class_index
            + float(
                torch.randn(
                    1,
                    generator=generator
                ).item()
                * 7.0
            )
        )

        bmi = (
            23.0
            + 1.8
            * class_index
            + float(
                torch.randn(
                    1,
                    generator=generator
                ).item()
                * 3.0
            )
        )

        biomarker = (
            0.4
            + 0.7
            * class_index
            + float(
                torch.randn(
                    1,
                    generator=generator
                ).item()
                * 0.35
            )
        )

        sex = (
            "F"
            if (
                patient_index
                % 2
                == 0
            )
            else "M"
        )

        symptom_group = (
            symptom_options[
                (
                    patient_index
                    + class_index
                )
                % 3
            ]
        )

        site = sites[
            patient_index
            % len(
                sites
            )
        ]

        device = devices[
            (
                patient_index
                + patient_index // 4
            )
            % len(
                devices
            )
        ]

        # Introduce missing numerical values.
        if (
            patient_index
            % 11
            == 0
        ):
            bmi = float(
                "nan"
            )

        if (
            patient_index
            % 13
            == 0
        ):
            biomarker = float(
                "nan"
            )

        # Introduce missing categorical values.
        if (
            patient_index
            % 17
            == 0
        ):
            symptom_group = None

        for image_index in range(
            images_per_patient
        ):
            image = make_multimodal_image(
                class_index,
                image_size,
                generator
            )

            # Mild site/device appearance shift.
            if site == "Site_A":
                image = (
                    image
                    * 0.90
                    + 0.03
                ).clamp(
                    0.0,
                    1.0
                )

            elif site == "Site_C":
                image = (
                    image
                    * 1.08
                ).clamp(
                    0.0,
                    1.0
                )

            if device == "Device_C":
                image = (
                    F.avg_pool2d(
                        image.unsqueeze(
                            0
                        ),
                        kernel_size=3,
                        stride=1,
                        padding=1
                    )
                    .squeeze(
                        0
                    )
                )

            tensor_index = len(
                images
            )

            images.append(
                image
            )

            records.append({
                "tensor_index":
                    tensor_index,

                "patient_id":
                    patient_id,

                "image_index":
                    image_index,

                "label":
                    class_index,

                "age":
                    age,

                "bmi":
                    bmi,

                "biomarker":
                    biomarker,

                "sex":
                    sex,

                "symptom_group":
                    symptom_group,

                "site":
                    site,

                "device":
                    device
            })

    return (
        images,
        pd.DataFrame(
            records
        )
    )


multimodal_images, multimodal_metadata = (
    create_multimodal_patient_data()
)

print(
    "Images:",
    len(
        multimodal_images
    )
)

print(
    "Patients:",
    multimodal_metadata[
        "patient_id"
    ].nunique()
)


# 9. Inspect the Metadata


In [ ]:
print(
    multimodal_metadata.head()
)

print()

print(
    multimodal_metadata.isna().sum()
)


# 10. Split at the Patient Level

Clinical variables are repeated across images from one patient.

Therefore we must still split by:

$$
\boxed{
Patient
}
$$

not image.


In [ ]:
patient_table = (
    multimodal_metadata
    .groupby(
        "patient_id",
        as_index=False
    )
    .agg({
        "label":
            "first"
    })
)

train_ids = []
val_ids = []
test_ids = []

for class_index in sorted(
    patient_table[
        "label"
    ].unique()
):
    ids = (
        patient_table[
            patient_table[
                "label"
            ]
            == class_index
        ][
            "patient_id"
        ]
        .tolist()
    )

    random.Random(
        42
        + int(
            class_index
        )
    ).shuffle(
        ids
    )

    n = len(
        ids
    )

    n_train = int(
        0.70
        * n
    )

    n_val = int(
        0.15
        * n
    )

    train_ids.extend(
        ids[
            :n_train
        ]
    )

    val_ids.extend(
        ids[
            n_train:
            n_train + n_val
        ]
    )

    test_ids.extend(
        ids[
            n_train + n_val:
        ]
    )

train_ids = set(
    train_ids
)

val_ids = set(
    val_ids
)

test_ids = set(
    test_ids
)

assert train_ids.isdisjoint(
    val_ids
)

assert train_ids.isdisjoint(
    test_ids
)

assert val_ids.isdisjoint(
    test_ids
)

print(
    len(train_ids),
    len(val_ids),
    len(test_ids)
)


# 11. Create Image-Level Split Tables


In [ ]:
train_df = (
    multimodal_metadata[
        multimodal_metadata[
            "patient_id"
        ].isin(
            train_ids
        )
    ]
    .reset_index(
        drop=True
    )
)

val_df = (
    multimodal_metadata[
        multimodal_metadata[
            "patient_id"
        ].isin(
            val_ids
        )
    ]
    .reset_index(
        drop=True
    )
)

test_df = (
    multimodal_metadata[
        multimodal_metadata[
            "patient_id"
        ].isin(
            test_ids
        )
    ]
    .reset_index(
        drop=True
    )
)

print(
    len(train_df),
    len(val_df),
    len(test_df)
)


# 12. Clinical Feature Preprocessing

Numerical variables usually need:

- Missing-value handling
- Scaling

Categorical variables usually need:

- Missing category handling
- Vocabulary mapping
- Embedding or one-hot representation

The key rule is:

$$
\boxed{
Fit\ Preprocessing\ on\ Training\ Only
}
$$


# 13. Why Standardize Numerical Features?

Suppose:

$$
age\approx60
$$

while:

$$
biomarker\approx0.8
$$

Different scales can make optimization harder.

Standardization:

$$
\boxed{
x'
=
\frac{x-\mu}{\sigma}
}
$$


# 14. Missing Numerical Values

Common strategies include:

- Median imputation
- Mean imputation
- Learned imputation
- Missingness indicator
- Model architectures designed for missing data

A simple robust baseline is:

> **Training median + missingness indicator**


# 15. Why Add a Missingness Indicator?

If BMI is missing, replacing it with the median alone makes:

> Missing BMI

look identical to:

> A real BMI equal to the median.

Add a mask:

$$
m_{BMI}
=
\begin{cases}
1 & \text{missing}\\
0 & \text{observed}
\end{cases}
$$


# 16. Categorical Variables

A categorical variable such as:

```text
sex = F / M
```

should not be treated as a continuous number like:

$$
F=0,\ M=1
$$

unless that numerical meaning is explicitly justified.

Neural networks commonly use:

> **Embeddings**


# 17. Unknown and Missing Categories

For deployment, validation/test may contain a category not seen during training.

Use reserved tokens such as:

```text
__UNK__
__MISSING__
```

This avoids crashing when unseen categories appear.


# 18. Build a Training-Only Clinical Preprocessor


In [ ]:
class ClinicalPreprocessor:
    def __init__(
        self,
        numerical_columns,
        categorical_columns
    ):
        self.numerical_columns = (
            list(
                numerical_columns
            )
        )

        self.categorical_columns = (
            list(
                categorical_columns
            )
        )

        self.numeric_medians = {}
        self.numeric_means = {}
        self.numeric_stds = {}
        self.category_maps = {}

    def fit(
        self,
        dataframe
    ):
        for column in (
            self.numerical_columns
        ):
            observed = dataframe[
                column
            ].dropna()

            median = float(
                observed.median()
            )

            filled = dataframe[
                column
            ].fillna(
                median
            )

            mean = float(
                filled.mean()
            )

            std = float(
                filled.std()
            )

            if std < 1e-8:
                std = 1.0

            self.numeric_medians[
                column
            ] = median

            self.numeric_means[
                column
            ] = mean

            self.numeric_stds[
                column
            ] = std

        for column in (
            self.categorical_columns
        ):
            values = (
                dataframe[
                    column
                ]
                .fillna(
                    "__MISSING__"
                )
                .astype(
                    str
                )
                .unique()
                .tolist()
            )

            mapping = {
                "__UNK__":
                    0,

                "__MISSING__":
                    1
            }

            next_index = 2

            for value in sorted(
                values
            ):
                if value not in mapping:
                    mapping[
                        value
                    ] = (
                        next_index
                    )

                    next_index += 1

            self.category_maps[
                column
            ] = mapping

        return self


# 19. Fit Only on Training Patients


In [ ]:
numerical_columns = [
    "age",
    "bmi",
    "biomarker"
]

categorical_columns = [
    "sex",
    "symptom_group"
]

clinical_preprocessor = (
    ClinicalPreprocessor(
        numerical_columns,
        categorical_columns
    )
    .fit(
        train_df
    )
)

print(
    clinical_preprocessor
    .numeric_medians
)

print(
    clinical_preprocessor
    .category_maps
)


# 20. Transform One Clinical Row

The numerical vector will contain:

- Standardized values
- Missingness indicators

For three numerical variables:

$$
3\ values
+
3\ missing\ indicators
=
6
$$

numerical inputs.


In [ ]:
def transform_clinical_row(
    row,
    preprocessor
):
    numeric_values = []
    missing_indicators = []

    for column in (
        preprocessor
        .numerical_columns
    ):
        value = row[
            column
        ]

        is_missing = (
            pd.isna(
                value
            )
        )

        if is_missing:
            value = (
                preprocessor
                .numeric_medians[
                    column
                ]
            )

        standardized = (
            float(
                value
            )
            -
            preprocessor
            .numeric_means[
                column
            ]
        ) / (
            preprocessor
            .numeric_stds[
                column
            ]
        )

        numeric_values.append(
            standardized
        )

        missing_indicators.append(
            float(
                is_missing
            )
        )

    numeric_tensor = torch.tensor(
        numeric_values
        + missing_indicators,
        dtype=torch.float32
    )

    categorical_indices = []

    for column in (
        preprocessor
        .categorical_columns
    ):
        value = row[
            column
        ]

        if pd.isna(
            value
        ):
            value = (
                "__MISSING__"
            )

        else:
            value = str(
                value
            )

        mapping = (
            preprocessor
            .category_maps[
                column
            ]
        )

        categorical_indices.append(
            mapping.get(
                value,
                mapping[
                    "__UNK__"
                ]
            )
        )

    categorical_tensor = torch.tensor(
        categorical_indices,
        dtype=torch.long
    )

    return (
        numeric_tensor,
        categorical_tensor
    )


In [ ]:
example_numeric, example_categorical = (
    transform_clinical_row(
        train_df.iloc[
            0
        ],
        clinical_preprocessor
    )
)

print(
    "Numerical:",
    example_numeric,
    example_numeric.shape
)

print(
    "Categorical:",
    example_categorical,
    example_categorical.shape
)


# 21. Image Normalization

Image preprocessing must also be fitted from training data only.

For this synthetic grayscale dataset we calculate:

$$
\mu_{image,train}
$$

and:

$$
\sigma_{image,train}
$$


In [ ]:
train_image_stack = torch.stack([
    multimodal_images[
        int(
            index
        )
    ]
    for index in train_df[
        "tensor_index"
    ].tolist()
])

image_train_mean = float(
    train_image_stack.mean().item()
)

image_train_std = float(
    train_image_stack.std().item()
)

print(
    image_train_mean,
    image_train_std
)


# 22. Multimodal Dataset

Each sample will return:

- Image tensor
- Numerical tensor
- Categorical tensor
- Target
- Row index

This lets one DataLoader feed both encoders.


In [ ]:
class MultimodalDataset(
    Dataset
):
    def __init__(
        self,
        dataframe,
        images,
        clinical_preprocessor,
        image_mean,
        image_std,
        training=False
    ):
        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.images = images
        self.clinical_preprocessor = (
            clinical_preprocessor
        )

        self.image_mean = (
            image_mean
        )

        self.image_std = (
            image_std
        )

        self.training = (
            training
        )

    def __len__(
        self
    ):
        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index
    ):
        row = self.dataframe.iloc[
            index
        ]

        image = self.images[
            int(
                row[
                    "tensor_index"
                ]
            )
        ].clone()

        if self.training:
            shift = int(
                torch.randint(
                    -2,
                    3,
                    (1,)
                ).item()
            )

            image = torch.roll(
                image,
                shifts=shift,
                dims=2
            )

        image = (
            image
            - self.image_mean
        ) / (
            self.image_std
            + 1e-8
        )

        numeric, categorical = (
            transform_clinical_row(
                row,
                self.clinical_preprocessor
            )
        )

        target = torch.tensor(
            int(
                row[
                    "label"
                ]
            ),
            dtype=torch.long
        )

        return (
            image,
            numeric,
            categorical,
            target,
            index
        )


# 23. Create Datasets and DataLoaders


In [ ]:
train_dataset = MultimodalDataset(
    train_df,
    multimodal_images,
    clinical_preprocessor,
    image_train_mean,
    image_train_std,
    training=True
)

val_dataset = MultimodalDataset(
    val_df,
    multimodal_images,
    clinical_preprocessor,
    image_train_mean,
    image_train_std,
    training=False
)

test_dataset = MultimodalDataset(
    test_df,
    multimodal_images,
    clinical_preprocessor,
    image_train_mean,
    image_train_std,
    training=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)


# 24. Inspect One Multimodal Batch

Always inspect shapes before building the model.


In [ ]:
(
    batch_images,
    batch_numeric,
    batch_categorical,
    batch_targets,
    batch_indices
) = next(
    iter(
        train_loader
    )
)

print(
    "Images:",
    batch_images.shape
)

print(
    "Numerical:",
    batch_numeric.shape
)

print(
    "Categorical:",
    batch_categorical.shape
)

print(
    "Targets:",
    batch_targets.shape
)


# 25. Shape Reasoning

For batch size:

$$
N=32
$$

we have:

$$
\begin{array}{|c|c|}
\hline
Image & (32,\ 1,\ 48,\ 48) \\
\hline
Numerical & (32,\ 6) \\
\hline
Categorical & (32,\ 2) \\
\hline
Target & (32) \\
\hline
\end{array}
$$


# 26. Image Encoder

An image encoder converts:

$$
(N,\ 1,\ H,\ W)
$$

into:

$$
(N,\ D_{image})
$$

We will use:

$$
D_{image}=64
$$


In [ ]:
class ImageEncoder(nn.Module):
    def __init__(
        self,
        output_dim=64
    ):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                1,
                16,
                3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                2
            ),

            nn.Conv2d(
                16,
                32,
                3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(
                2
            ),

            nn.Conv2d(
                32,
                64,
                3,
                padding=1
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(
                1
            )
        )

        self.projection = nn.Linear(
            64,
            output_dim
        )

    def forward(
        self,
        image
    ):
        x = self.features(
            image
        )

        x = torch.flatten(
            x,
            start_dim=1
        )

        return self.projection(
            x
        )


# 27. Image Encoder Shape


In [ ]:
image_encoder = ImageEncoder(
    output_dim=64
)

image_features = image_encoder(
    batch_images
)

print(
    image_features.shape
)


# 28. Tabular Encoder

The tabular encoder must combine:

- Numerical features
- Categorical embeddings

For categorical variable $j$:

$$
category_j
\rightarrow
Embedding_j
$$

Then all clinical features are concatenated.


# 29. Determine Category Vocabulary Sizes


In [ ]:
category_cardinalities = [
    len(
        clinical_preprocessor
        .category_maps[
            column
        ]
    )
    for column in categorical_columns
]

print(
    category_cardinalities
)


# 30. Embedding Dimension

For small categorical variables, a small embedding such as:

$$
4
$$

dimensions is enough for demonstration.

Example:

$$
sex\ index
\rightarrow
(4)
$$


In [ ]:
class TabularEncoder(nn.Module):
    def __init__(
        self,
        numeric_dim,
        category_cardinalities,
        embedding_dim=4,
        output_dim=32
    ):
        super().__init__()

        self.embeddings = (
            nn.ModuleList([
                nn.Embedding(
                    cardinality,
                    embedding_dim
                )
                for cardinality in (
                    category_cardinalities
                )
            ])
        )

        combined_dim = (
            numeric_dim
            + embedding_dim
            * len(
                category_cardinalities
            )
        )

        self.network = nn.Sequential(
            nn.Linear(
                combined_dim,
                64
            ),
            nn.ReLU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                64,
                output_dim
            ),
            nn.ReLU()
        )

    def forward(
        self,
        numeric,
        categorical
    ):
        embedded = []

        for index, embedding in enumerate(
            self.embeddings
        ):
            embedded.append(
                embedding(
                    categorical[
                        :,
                        index
                    ]
                )
            )

        if embedded:
            categorical_features = (
                torch.cat(
                    embedded,
                    dim=1
                )
            )

            x = torch.cat(
                [
                    numeric,
                    categorical_features
                ],
                dim=1
            )

        else:
            x = numeric

        return self.network(
            x
        )


# 31. Tabular Encoder Shape


In [ ]:
numeric_dim = (
    len(
        numerical_columns
    )
    * 2
)

tabular_encoder = (
    TabularEncoder(
        numeric_dim=numeric_dim,
        category_cardinalities=(
            category_cardinalities
        ),
        embedding_dim=4,
        output_dim=32
    )
)

tabular_features = (
    tabular_encoder(
        batch_numeric,
        batch_categorical
    )
)

print(
    tabular_features.shape
)


# 32. Feature Fusion

Now we have:

$$
ImageFeatures:
(N,\ 64)
$$

and:

$$
ClinicalFeatures:
(N,\ 32)
$$

Concatenation gives:

$$
\boxed{
(N,\ 96)
}
$$


In [ ]:
fused_features = torch.cat(
    [
        image_features,
        tabular_features
    ],
    dim=1
)

print(
    fused_features.shape
)


# 33. Concatenation-Based Multimodal Network

This is one of the simplest and strongest multimodal baselines:

$$
Image
\rightarrow
f_{image}
$$

$$
Clinical
\rightarrow
f_{tab}
$$

then:

$$
[f_{image};f_{tab}]
\rightarrow
MLP
\rightarrow
Logits
$$


In [ ]:
class MultimodalFusionModel(
    nn.Module
):
    def __init__(
        self,
        numeric_dim,
        category_cardinalities,
        num_classes=3
    ):
        super().__init__()

        self.image_encoder = (
            ImageEncoder(
                output_dim=64
            )
        )

        self.tabular_encoder = (
            TabularEncoder(
                numeric_dim=(
                    numeric_dim
                ),
                category_cardinalities=(
                    category_cardinalities
                ),
                embedding_dim=4,
                output_dim=32
            )
        )

        self.fusion_head = (
            nn.Sequential(
                nn.Linear(
                    64 + 32,
                    64
                ),
                nn.ReLU(),
                nn.Dropout(
                    0.2
                ),
                nn.Linear(
                    64,
                    num_classes
                )
            )
        )

    def forward(
        self,
        image,
        numeric,
        categorical
    ):
        image_features = (
            self.image_encoder(
                image
            )
        )

        tabular_features = (
            self.tabular_encoder(
                numeric,
                categorical
            )
        )

        fused = torch.cat(
            [
                image_features,
                tabular_features
            ],
            dim=1
        )

        return self.fusion_head(
            fused
        )


# 34. Verify Fused Output Shape


In [ ]:
fusion_model = MultimodalFusionModel(
    numeric_dim=numeric_dim,
    category_cardinalities=(
        category_cardinalities
    ),
    num_classes=3
)

fusion_logits = fusion_model(
    batch_images,
    batch_numeric,
    batch_categorical
)

print(
    fusion_logits.shape
)


# 35. Early Fusion

**Early fusion** combines modalities relatively early.

Examples:

- Concatenate raw tabular features with early image features
- Encode metadata into channels/tokens
- Joint token sequence in a transformer

Advantages:

- Rich cross-modal interaction

Disadvantages:

- Harder architecture
- Different scales/formats
- Higher overfitting risk


# 36. Intermediate Fusion

The model we implemented is often called:

> **Intermediate feature fusion**

because each modality is encoded separately first.

Then learned feature vectors are concatenated.

This is a very useful practical baseline.


# 37. Late Fusion

Late fusion combines predictions instead of intermediate features.

Example:

$$
p_{image}
$$

and:

$$
p_{tabular}
$$

then:

$$
p_{final}
=
\alpha p_{image}
+
(1-\alpha)p_{tabular}
$$

or combine logits.


# 38. Why Late Fusion Is Useful

Advantages:

- Each modality model is independent
- Easy to debug
- Easy to handle missing modalities
- Existing models can be reused

Disadvantages:

- Limited cross-modal feature interaction


# 39. Image-Only Baseline


In [ ]:
class ImageOnlyClassifier(
    nn.Module
):
    def __init__(
        self,
        num_classes=3
    ):
        super().__init__()

        self.encoder = (
            ImageEncoder(
                output_dim=64
            )
        )

        self.classifier = nn.Linear(
            64,
            num_classes
        )

    def forward(
        self,
        image
    ):
        features = self.encoder(
            image
        )

        return self.classifier(
            features
        )


# 40. Tabular-Only Baseline


In [ ]:
class TabularOnlyClassifier(
    nn.Module
):
    def __init__(
        self,
        numeric_dim,
        category_cardinalities,
        num_classes=3
    ):
        super().__init__()

        self.encoder = (
            TabularEncoder(
                numeric_dim,
                category_cardinalities,
                embedding_dim=4,
                output_dim=32
            )
        )

        self.classifier = nn.Linear(
            32,
            num_classes
        )

    def forward(
        self,
        numeric,
        categorical
    ):
        features = self.encoder(
            numeric,
            categorical
        )

        return self.classifier(
            features
        )


# 41. Late-Fusion Model


In [ ]:
class LateFusionModel(
    nn.Module
):
    def __init__(
        self,
        numeric_dim,
        category_cardinalities,
        num_classes=3,
        image_weight=0.5
    ):
        super().__init__()

        self.image_model = (
            ImageOnlyClassifier(
                num_classes
            )
        )

        self.tabular_model = (
            TabularOnlyClassifier(
                numeric_dim,
                category_cardinalities,
                num_classes
            )
        )

        self.image_weight = (
            image_weight
        )

    def forward(
        self,
        image,
        numeric,
        categorical
    ):
        image_logits = (
            self.image_model(
                image
            )
        )

        tabular_logits = (
            self.tabular_model(
                numeric,
                categorical
            )
        )

        return (
            self.image_weight
            * image_logits
            +
            (
                1.0
                - self.image_weight
            )
            * tabular_logits
        )


# 42. Early vs Intermediate vs Late Fusion

$$
\begin{array}{|c|c|}
\hline
\textbf{Fusion} & \textbf{Where Information Combines} \\
\hline
Early & Near\ raw/low-level\ inputs \\
\hline
Intermediate & Learned\ feature\ vectors \\
\hline
Late & Predictions/logits \\
\hline
\end{array}
$$

There is no universal best choice.

Compare experimentally.


# 43. Device Setup


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device
)


# 44. Reproducibility Helper


In [ ]:
def set_seed(
    seed
):
    random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


# 45. Generic Multimodal Training Loop

The training loop receives all modalities together.

The forward function depends on model type.


In [ ]:
def multimodal_forward(
    model,
    mode,
    image,
    numeric,
    categorical
):
    if mode == "image_only":
        return model(
            image
        )

    if mode == "tabular_only":
        return model(
            numeric,
            categorical
        )

    if mode in {
        "fusion",
        "late_fusion"
    }:
        return model(
            image,
            numeric,
            categorical
        )

    raise ValueError(
        f"Unknown mode: {mode}"
    )


In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    mode
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for (
        image,
        numeric,
        categorical,
        target,
        _
    ) in loader:
        image = image.to(
            device
        )

        numeric = numeric.to(
            device
        )

        categorical = (
            categorical.to(
                device
            )
        )

        target = target.to(
            device
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = multimodal_forward(
            model,
            mode,
            image,
            numeric,
            categorical
        )

        loss = criterion(
            logits,
            target
        )

        loss.backward()
        optimizer.step()

        batch_size = (
            target.size(0)
        )

        total_loss += (
            loss.item()
            * batch_size
        )

        total_correct += (
            (
                logits.argmax(
                    dim=1
                )
                == target
            )
            .sum()
            .item()
        )

        total_samples += (
            batch_size
        )

    return (
        total_loss
        / total_samples,
        total_correct
        / total_samples
    )


# 46. Validation Loop


In [ ]:
def evaluate_model(
    model,
    loader,
    criterion,
    device,
    mode
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.inference_mode():
        for (
            image,
            numeric,
            categorical,
            target,
            _
        ) in loader:
            image = image.to(
                device
            )

            numeric = numeric.to(
                device
            )

            categorical = (
                categorical.to(
                    device
                )
            )

            target = target.to(
                device
            )

            logits = multimodal_forward(
                model,
                mode,
                image,
                numeric,
                categorical
            )

            loss = criterion(
                logits,
                target
            )

            batch_size = (
                target.size(0)
            )

            total_loss += (
                loss.item()
                * batch_size
            )

            total_correct += (
                (
                    logits.argmax(
                        dim=1
                    )
                    == target
                )
                .sum()
                .item()
            )

            total_samples += (
                batch_size
            )

    return (
        total_loss
        / total_samples,
        total_correct
        / total_samples
    )


# 47. Best-Checkpoint Training


In [ ]:
def fit_model(
    model,
    train_loader,
    val_loader,
    mode,
    device,
    epochs=6,
    lr=1e-3
):
    criterion = (
        nn.CrossEntropyLoss()
    )

    optimizer = (
        torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=1e-4
        )
    )

    best_val_loss = float(
        "inf"
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(
        epochs
    ):
        train_loss, train_acc = (
            train_one_epoch(
                model,
                train_loader,
                criterion,
                optimizer,
                device,
                mode
            )
        )

        val_loss, val_acc = (
            evaluate_model(
                model,
                val_loader,
                criterion,
                device,
                mode
            )
        )

        history[
            "train_loss"
        ].append(
            train_loss
        )

        history[
            "train_acc"
        ].append(
            train_acc
        )

        history[
            "val_loss"
        ].append(
            val_loss
        )

        history[
            "val_acc"
        ].append(
            val_acc
        )

        if val_loss < best_val_loss:
            best_val_loss = (
                val_loss
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

    model.load_state_dict(
        best_state
    )

    return (
        model,
        history,
        best_val_loss
    )


# 48. Train Image-Only Baseline


In [ ]:
set_seed(
    42
)

image_only_model = (
    ImageOnlyClassifier(
        num_classes=3
    )
    .to(
        device
    )
)

image_only_model, image_history, image_best_loss = (
    fit_model(
        image_only_model,
        train_loader,
        val_loader,
        mode="image_only",
        device=device,
        epochs=5
    )
)

print(
    "Image-only best val loss:",
    image_best_loss
)


# 49. Train Tabular-Only Baseline


In [ ]:
set_seed(
    42
)

tabular_only_model = (
    TabularOnlyClassifier(
        numeric_dim,
        category_cardinalities,
        num_classes=3
    )
    .to(
        device
    )
)

tabular_only_model, tabular_history, tabular_best_loss = (
    fit_model(
        tabular_only_model,
        train_loader,
        val_loader,
        mode="tabular_only",
        device=device,
        epochs=5
    )
)

print(
    "Tabular-only best val loss:",
    tabular_best_loss
)


# 50. Train Feature-Fusion Model


In [ ]:
set_seed(
    42
)

fusion_model = (
    MultimodalFusionModel(
        numeric_dim,
        category_cardinalities,
        num_classes=3
    )
    .to(
        device
    )
)

fusion_model, fusion_history, fusion_best_loss = (
    fit_model(
        fusion_model,
        train_loader,
        val_loader,
        mode="fusion",
        device=device,
        epochs=5
    )
)

print(
    "Fusion best val loss:",
    fusion_best_loss
)


# 51. Train Late-Fusion Model


In [ ]:
set_seed(
    42
)

late_fusion_model = (
    LateFusionModel(
        numeric_dim,
        category_cardinalities,
        num_classes=3,
        image_weight=0.5
    )
    .to(
        device
    )
)

late_fusion_model, late_history, late_best_loss = (
    fit_model(
        late_fusion_model,
        train_loader,
        val_loader,
        mode="late_fusion",
        device=device,
        epochs=5
    )
)

print(
    "Late-fusion best val loss:",
    late_best_loss
)


# 52. Compare Validation Models

This is the central multimodal ablation:

$$
Image\ Only
$$

vs:

$$
Tabular\ Only
$$

vs:

$$
Image+Clinical
$$


In [ ]:
criterion = nn.CrossEntropyLoss()

comparison_rows = []

for name, model, mode in [
    (
        "image_only",
        image_only_model,
        "image_only"
    ),
    (
        "tabular_only",
        tabular_only_model,
        "tabular_only"
    ),
    (
        "feature_fusion",
        fusion_model,
        "fusion"
    ),
    (
        "late_fusion",
        late_fusion_model,
        "late_fusion"
    )
]:
    val_loss, val_accuracy = (
        evaluate_model(
            model,
            val_loader,
            criterion,
            device,
            mode
        )
    )

    comparison_rows.append({
        "model":
            name,

        "val_loss":
            val_loss,

        "val_accuracy":
            val_accuracy
    })

comparison_df = pd.DataFrame(
    comparison_rows
)

print(
    comparison_df
)


# 53. Modality Ablation Studies

A modality ablation asks:

> What happens if one modality is removed?

Examples:

- Image only
- Clinical only
- Image + clinical
- Image + numerical only
- Image + categorical only

This reveals where predictive information comes from.


# 54. Why Fusion Improvement Must Be Interpreted Carefully

Suppose:

$$
Image\ Only=0.80
$$

and:

$$
Fusion=0.92
$$

Possible explanations:

- Clinical variables provide complementary information
- Clinical variables contain leakage
- Site/device variables indirectly encode label
- Missingness itself is highly predictive
- Small validation set produced chance improvement

Always investigate.


# 55. Leakage From Clinical Variables

Potential leakage variables include:

- Post-diagnosis treatment
- Pathology-confirmed status
- Radiology conclusion
- Future lab result
- Surgical result
- Follow-up outcome

A multimodal model can achieve spectacular but meaningless performance if these are included.


# 56. Leakage From Missingness

Missingness patterns can themselves encode workflow.

Example:

> Biomarker measured only when clinicians strongly suspect disease.

Then:

$$
BiomarkerMissing
$$

may encode clinical suspicion.

That may or may not be valid depending on deployment.

You must decide whether the missingness pattern will exist at prediction time.


# 57. Site and Device as Clinical Inputs

Site/device metadata can be useful for:

- Harmonization
- Auditing
- Subgroup evaluation

But directly giving site/device to the classifier may encourage shortcuts.

If included, justify why they should influence the clinical prediction.


# 58. Clinical Preprocessing Must Be Fold-Specific in Cross-Validation

For patient-grouped CV:

For every fold:

1. Fit numerical medians on training patients
2. Fit numerical mean/std on training patients
3. Fit categorical vocabularies on training patients
4. Apply frozen mappings to validation patients

This is exactly analogous to image normalization leakage.


# 59. Unknown Categories in External Data

Suppose training contains:

```text
symptom_group = A, B, C
```

but external site contains:

```text
symptom_group = D
```

The external category must map to:

```text
__UNK__
```

Do not refit the vocabulary on the external test set.


# 60. Collect Multimodal Predictions


In [ ]:
def collect_predictions(
    model,
    dataset,
    loader,
    mode,
    device
):
    model.eval()

    rows = []

    with torch.inference_mode():
        for (
            image,
            numeric,
            categorical,
            target,
            indices
        ) in loader:
            image = image.to(
                device
            )

            numeric = numeric.to(
                device
            )

            categorical = (
                categorical.to(
                    device
                )
            )

            logits = multimodal_forward(
                model,
                mode,
                image,
                numeric,
                categorical
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            ).cpu()

            predictions = logits.argmax(
                dim=1
            ).cpu()

            for position in range(
                len(
                    indices
                )
            ):
                local_index = int(
                    indices[
                        position
                    ].item()
                )

                row = (
                    dataset
                    .dataframe
                    .iloc[
                        local_index
                    ]
                )

                record = {
                    "patient_id":
                        row[
                            "patient_id"
                        ],

                    "site":
                        row[
                            "site"
                        ],

                    "device":
                        row[
                            "device"
                        ],

                    "true_label":
                        int(
                            target[
                                position
                            ].item()
                        ),

                    "predicted_label":
                        int(
                            predictions[
                                position
                            ].item()
                        )
                }

                for class_index in range(
                    probabilities.shape[
                        1
                    ]
                ):
                    record[
                        f"prob_class_{class_index}"
                    ] = float(
                        probabilities[
                            position,
                            class_index
                        ].item()
                    )

                rows.append(
                    record
                )

    return pd.DataFrame(
        rows
    )


# 61. Patient-Level Multimodal Evaluation

Each patient has multiple images but the same clinical variables.

If the final decision is per patient:

1. Predict each image
2. Average probabilities across images
3. Produce one patient-level prediction


In [ ]:
def aggregate_patient_predictions(
    prediction_df,
    num_classes=3
):
    probability_columns = [
        f"prob_class_{index}"
        for index in range(
            num_classes
        )
    ]

    rows = []

    for patient_id, group in (
        prediction_df.groupby(
            "patient_id"
        )
    ):
        true_labels = (
            group[
                "true_label"
            ]
            .unique()
        )

        assert len(
            true_labels
        ) == 1

        mean_probs = (
            group[
                probability_columns
            ]
            .mean()
        )

        predicted_label = int(
            mean_probs
            .to_numpy()
            .argmax()
        )

        record = {
            "patient_id":
                patient_id,

            "true_label":
                int(
                    true_labels[
                        0
                    ]
                ),

            "predicted_label":
                predicted_label
        }

        for column in (
            probability_columns
        ):
            record[
                column
            ] = float(
                mean_probs[
                    column
                ]
            )

        rows.append(
            record
        )

    return pd.DataFrame(
        rows
    )


# 62. Patient-Level Validation Comparison


In [ ]:
patient_comparison_rows = []

for name, model, mode in [
    (
        "image_only",
        image_only_model,
        "image_only"
    ),
    (
        "tabular_only",
        tabular_only_model,
        "tabular_only"
    ),
    (
        "feature_fusion",
        fusion_model,
        "fusion"
    ),
    (
        "late_fusion",
        late_fusion_model,
        "late_fusion"
    )
]:
    image_level = (
        collect_predictions(
            model,
            val_dataset,
            val_loader,
            mode,
            device
        )
    )

    patient_level = (
        aggregate_patient_predictions(
            image_level,
            num_classes=3
        )
    )

    accuracy = (
        patient_level[
            "true_label"
        ].to_numpy()
        ==
        patient_level[
            "predicted_label"
        ].to_numpy()
    ).mean()

    patient_comparison_rows.append({
        "model":
            name,

        "patient_val_accuracy":
            float(
                accuracy
            )
    })

patient_comparison_df = (
    pd.DataFrame(
        patient_comparison_rows
    )
)

print(
    patient_comparison_df
)


# 63. Repeated Clinical Information Across Images

If a patient has:

$$
10
$$

images,

the same clinical vector is repeated:

$$
10
$$

times in image-level training.

That can overweight patients with more images.

Possible solutions:

- Sample one image per patient per epoch
- Patient-balanced sampling
- Study-level model
- Aggregate images first
- Multi-instance learning


# 64. Patient-Balanced Sampling Intuition

If patient A has:

$$
20
$$

images

and patient B has:

$$
2
$$

images,

ordinary image sampling gives patient A about:

$$
10\times
$$

more influence.

This may be undesirable for patient-level tasks.


# 65. Handling Missing Modalities

Real multimodal datasets may contain:

- Image available, clinical data missing
- Clinical data available, image missing
- Some clinical variables missing
- Entire modality missing

A model should have an explicit strategy.


# 66. Missing Clinical Variables vs Missing Clinical Modality

These are different.

## Missing variable

Example:

```text
BMI missing
```

Other clinical features still exist.

## Missing modality

Example:

```text
No clinical record available
```

The whole clinical branch may be unavailable.


# 67. Modality Presence Masks

A useful model input includes:

$$
m_{image}
$$

and:

$$
m_{tabular}
$$

where:

$$
1=\text{present}
$$

$$
0=\text{missing}
$$

This tells the network whether a zero feature vector represents real information or missing modality.


# 68. Missing-Modality Fusion Model

We can multiply each encoded modality by a presence mask and append the masks to the fusion head.


In [ ]:
class MissingAwareFusionModel(
    nn.Module
):
    def __init__(
        self,
        numeric_dim,
        category_cardinalities,
        num_classes=3
    ):
        super().__init__()

        self.image_encoder = (
            ImageEncoder(
                output_dim=64
            )
        )

        self.tabular_encoder = (
            TabularEncoder(
                numeric_dim,
                category_cardinalities,
                embedding_dim=4,
                output_dim=32
            )
        )

        self.head = nn.Sequential(
            nn.Linear(
                64
                + 32
                + 2,
                64
            ),
            nn.ReLU(),
            nn.Dropout(
                0.2
            ),
            nn.Linear(
                64,
                num_classes
            )
        )

    def forward(
        self,
        image,
        numeric,
        categorical,
        image_present,
        tabular_present
    ):
        image_features = (
            self.image_encoder(
                image
            )
        )

        tabular_features = (
            self.tabular_encoder(
                numeric,
                categorical
            )
        )

        image_features = (
            image_features
            * image_present
        )

        tabular_features = (
            tabular_features
            * tabular_present
        )

        fused = torch.cat(
            [
                image_features,
                tabular_features,
                image_present,
                tabular_present
            ],
            dim=1
        )

        return self.head(
            fused
        )


# 69. Presence Mask Shapes

For a batch of size:

$$
N
$$

use:

$$
image\_present:
(N,\ 1)
$$

$$
tabular\_present:
(N,\ 1)
$$


In [ ]:
batch_size = (
    batch_images.shape[
        0
    ]
)

image_present = torch.ones(
    batch_size,
    1
)

tabular_present = torch.ones(
    batch_size,
    1
)

missing_aware_model = (
    MissingAwareFusionModel(
        numeric_dim,
        category_cardinalities,
        num_classes=3
    )
)

missing_aware_logits = (
    missing_aware_model(
        batch_images,
        batch_numeric,
        batch_categorical,
        image_present,
        tabular_present
    )
)

print(
    missing_aware_logits.shape
)


# 70. Modality Dropout

A useful training strategy is:

> Randomly hide one modality during training.

This is called:

> **Modality dropout**

It can make the fused model less dependent on one modality.

But it should represent realistic missing-modality conditions.


In [ ]:
def sample_modality_masks(
    batch_size,
    image_drop_probability=0.1,
    tabular_drop_probability=0.1,
    device="cpu"
):
    image_present = (
        torch.rand(
            batch_size,
            1,
            device=device
        )
        >
        image_drop_probability
    ).float()

    tabular_present = (
        torch.rand(
            batch_size,
            1,
            device=device
        )
        >
        tabular_drop_probability
    ).float()

    both_missing = (
        (
            image_present
            + tabular_present
        )
        == 0
    )

    image_present[
        both_missing
    ] = 1.0

    return (
        image_present,
        tabular_present
    )


# 71. Why Prevent Both Modalities From Being Missing?

If the model receives:

$$
No\ Image
+
No\ Clinical\ Data
$$

there may be no meaningful basis for prediction.

The appropriate behavior may instead be:

> **Abstain / request data**


# 72. Missing-Modality Evaluation

A research-quality multimodal model should be evaluated under:

- Both modalities present
- Image only
- Tabular only
- Partially missing clinical variables

This tests graceful degradation.


# 73. Missing-Modality Late Fusion

Late fusion naturally supports missing modalities.

If only image is present:

$$
p=p_{image}
$$

If only tabular is present:

$$
p=p_{tabular}
$$

If both are present:

$$
p=\alpha p_{image}+(1-\alpha)p_{tabular}
$$


# 74. Modality Reliability Can Differ by Domain

On a new scanner:

- Image model may degrade
- Clinical model may remain stable

On a new hospital:

- Clinical prevalence/workflow may change
- Image model may remain stable

This is why domain-aware multimodal evaluation is valuable.


# 75. Site-Wise Multimodal Evaluation

For each site compare:

- Image-only
- Tabular-only
- Fusion

A fusion model that only improves one site may not be robust.


In [ ]:
def subgroup_accuracy(
    prediction_df,
    column
):
    rows = []

    for subgroup, group in (
        prediction_df.groupby(
            column
        )
    ):
        accuracy = (
            group[
                "true_label"
            ].to_numpy()
            ==
            group[
                "predicted_label"
            ].to_numpy()
        ).mean()

        rows.append({
            column:
                subgroup,

            "n":
                len(
                    group
                ),

            "accuracy":
                float(
                    accuracy
                )
        })

    return pd.DataFrame(
        rows
    )


In [ ]:
fusion_val_predictions = (
    collect_predictions(
        fusion_model,
        val_dataset,
        val_loader,
        mode="fusion",
        device=device
    )
)

print(
    subgroup_accuracy(
        fusion_val_predictions,
        "site"
    )
)


# 76. Clinical Feature Leakage Across Sites

A clinical variable may indirectly encode site.

Examples:

- Site-specific test naming
- Lab measurement units
- Missingness pattern
- Workflow-dependent variable
- Device-specific metadata

Always inspect feature distributions by site.


# 77. Numerical Features by Site


In [ ]:
patient_level_train = (
    train_df
    .drop_duplicates(
        "patient_id"
    )
)

print(
    patient_level_train.groupby(
        "site"
    )[
        numerical_columns
    ].mean()
)


# 78. Categorical Features by Site


In [ ]:
print(
    pd.crosstab(
        patient_level_train[
            "site"
        ],
        patient_level_train[
            "symptom_group"
        ],
        normalize="index"
    )
)


# 79. Clinical Feature Selection Should Be Scientific

Do not include every available column automatically.

For each candidate feature ask:

1. Is it available at prediction time?
2. Is it measured consistently across sites?
3. Is it clinically plausible?
4. Is missingness meaningful?
5. Could it encode outcome leakage?
6. Will it exist at deployment?


# 80. Feature Standardization in External Validation

Do not calculate:

$$
\mu_{external}
$$

and:

$$
\sigma_{external}
$$

for numerical variables just because the external site has different values.

Use training-derived preprocessing unless test-time adaptation is explicitly part of the protocol.


# 81. Categorical Embeddings and Small Datasets

Embeddings introduce learnable parameters.

For a category with very few patients, an embedding can overfit.

Possible alternatives:

- One-hot encoding
- Group rare categories
- Stronger regularization
- Remove unreliable variable


# 82. Clinical Feature Interactions

A tabular MLP can learn interactions such as:

$$
Age
\times
Biomarker
$$

implicitly.

But small medical datasets can overfit these interactions.

Keep the tabular branch compact.


# 83. Fusion Capacity

A huge fusion head can memorize the development set.

A good baseline:

- Small image feature vector
- Small clinical feature vector
- One or two fusion layers
- Dropout / weight decay


# 84. Feature Magnitude Imbalance

Suppose image features have magnitude:

$$
10
$$

and tabular features:

$$
0.1
$$

The fusion layer may favor one modality.

Useful tools include:

- Normalization
- Projection layers
- BatchNorm/LayerNorm
- Balanced feature dimensions


# 85. Optional Feature Normalization

We can normalize encoded vectors before fusion:

```python
F.normalize(
    features,
    dim=1
)
```

This is an architectural choice that should be validated experimentally.


# 86. Gated Fusion Intuition

Instead of simple concatenation, a model can learn a gate:

$$
g\in[0,1]
$$

then:

$$
z
=
g\cdot z_{image}
+
(1-g)\cdot z_{tab}
$$

This allows dynamic modality weighting.


# 87. Attention-Based Fusion Intuition

More advanced multimodal architectures use:

- Cross-attention
- Transformer tokens
- Co-attention
- Modality-specific attention

These can model richer interactions but need more data and stronger validation.


# 88. Simple Concatenation Is an Important Baseline

Do not jump to multimodal transformers before establishing:

$$
\boxed{
Image\ Only
}
$$

$$
\boxed{
Tabular\ Only
}
$$

$$
\boxed{
Simple\ Concatenation
}
$$

A sophisticated model should beat a simple baseline under a fair protocol.


# 89. Model Selection

Choose multimodal architecture using validation/CV only.

Do not select:

- Fusion type
- Embedding size
- Clinical variables
- Missing-modality strategy

using final test performance.


# 90. Patient-Grouped Cross-Validation for Multimodal Models

Every fold must fit:

- Image normalization
- Clinical medians
- Clinical mean/std
- Categorical vocabularies
- Feature-selection decisions

from fold-training patients only.


# 91. Modality Ablation Must Use the Same Folds

If image-only and fusion models use different patient folds, performance differences are confounded by patient difficulty.

Use:

$$
Same\ Patients
+
Same\ Folds
+
Same\ Seeds
$$


# 92. Multiple Seeds

Multimodal models can be especially sensitive to initialization because two encoders must co-adapt.

Repeat across several predefined seeds for research-quality comparisons.


# 93. External Multimodal Validation

External validation may reveal:

- New scanner distribution
- Different clinical prevalence
- Different missingness patterns
- Unknown categories
- Different lab measurement ranges

Therefore evaluate each modality and fusion separately externally.


# 94. Modality-Specific Domain Shift

An external site may create:

$$
P_{external}(Image)
\neq
P_{train}(Image)
$$

and independently:

$$
P_{external}(Clinical)
\neq
P_{train}(Clinical)
$$

Multimodal robustness requires understanding both.


# 95. Image Robust, Clinical Shifted

Possible case:

- Ultrasound acquisition is standardized
- Clinical workflow differs

Then the tabular branch may cause the fusion model to degrade.

This is why:

> Fusion should not be assumed to improve every domain.


# 96. Clinical Robust, Image Shifted

Possible case:

- Clinical features are stable
- New ultrasound scanner changes appearance

Then the clinical modality may help stabilize the fused model.


# 97. Missingness Shift

Training site:

$$
10\%
$$

biomarker missing.

External site:

$$
60\%
$$

missing.

This is a distribution shift in the **missingness mechanism**.

Evaluate it explicitly.


# 98. Missingness Mechanisms — Intuition

Common statistical language includes:

- MCAR — missing completely at random
- MAR — missing at random conditional on observed variables
- MNAR — missingness depends on unobserved value or process

Deep-learning pipelines still need to reason about *why* values are missing.


# 99. Clinical Variables With Different Units

A lab variable may be recorded in:

- mg/dL
- mmol/L

across sites.

This is not something neural-network normalization should silently fix.

Units must be harmonized scientifically before modeling.


# 100. Categorical Definition Shift

One hospital may define:

```text
symptom_group = A/B/C
```

differently from another hospital.

Matching category names do not guarantee matching meaning.

Metadata harmonization can be as important as image harmonization.


# 101. Multimodal Harmonization

Multimodal harmonization may require:

## Image branch

- Scanner normalization
- Resolution standardization
- Domain robustness

## Clinical branch

- Unit harmonization
- Coding harmonization
- Missing-value policy
- Category mapping

Both must be leakage-safe.


# 102. Outcome-Blind Preprocessing

A preprocessing step should generally not use test labels.

Examples:

- Standardizing age using training data: okay
- Selecting features based on external test AUROC: leakage
- Choosing categories after seeing test outcomes: leakage


# 103. Final Test Evaluation

After all development decisions are frozen, evaluate the selected model once on the final test set.


In [ ]:
selected_mode = (
    "fusion"
)

selected_model = (
    fusion_model
)

test_loss, test_accuracy = (
    evaluate_model(
        selected_model,
        test_loader,
        criterion,
        device,
        selected_mode
    )
)

print(
    "Test loss:",
    test_loss
)

print(
    "Test image-level accuracy:",
    test_accuracy
)


# 104. Patient-Level Final Test Evaluation


In [ ]:
test_predictions = (
    collect_predictions(
        selected_model,
        test_dataset,
        test_loader,
        selected_mode,
        device
    )
)

patient_test_predictions = (
    aggregate_patient_predictions(
        test_predictions,
        num_classes=3
    )
)

patient_test_accuracy = (
    patient_test_predictions[
        "true_label"
    ].to_numpy()
    ==
    patient_test_predictions[
        "predicted_label"
    ].to_numpy()
).mean()

print(
    "Patient-level test accuracy:",
    patient_test_accuracy
)


# 105. Save Reproducible Multimodal Outputs

Save:

- Split manifest
- Image normalization
- Clinical medians
- Clinical mean/std
- Category vocabularies
- Model config
- Checkpoint
- Predictions
- Modality ablation results


In [ ]:
MULTIMODAL_DIR = Path(
    "multimodal_results"
)

MULTIMODAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

preprocessing_config = {
    "image_train_mean":
        image_train_mean,

    "image_train_std":
        image_train_std,

    "numerical_columns":
        numerical_columns,

    "categorical_columns":
        categorical_columns,

    "numeric_medians":
        clinical_preprocessor
        .numeric_medians,

    "numeric_means":
        clinical_preprocessor
        .numeric_means,

    "numeric_stds":
        clinical_preprocessor
        .numeric_stds,

    "category_maps":
        clinical_preprocessor
        .category_maps
}

(
    MULTIMODAL_DIR
    / "preprocessing.json"
).write_text(
    json.dumps(
        preprocessing_config,
        indent=2
    ),
    encoding="utf-8"
)

comparison_df.to_csv(
    MULTIMODAL_DIR
    / "modality_ablation.csv",
    index=False
)

test_predictions.to_csv(
    MULTIMODAL_DIR
    / "test_image_predictions.csv",
    index=False
)

patient_test_predictions.to_csv(
    MULTIMODAL_DIR
    / "test_patient_predictions.csv",
    index=False
)

torch.save(
    selected_model.state_dict(),
    MULTIMODAL_DIR
    / "selected_model.pth"
)

print(
    "Multimodal outputs saved."
)


# 106. What Should Be Stored With a Clinical Preprocessor?

For reproducibility:

$$
\begin{array}{|c|c|}
\hline
Numerical\ medians & Missing\ value\ imputation \\
\hline
Numerical\ means & Standardization \\
\hline
Numerical\ stds & Standardization \\
\hline
Category\ maps & Stable\ categorical\ encoding \\
\hline
Feature\ names & Correct\ feature\ order \\
\hline
\end{array}
$$


# 107. Feature Order Is Critical

If training order is:

```text
age, bmi, biomarker
```

but deployment order is:

```text
biomarker, age, bmi
```

the model silently receives wrong inputs.

Always save the feature order.


# 108. Categorical Column Order Is Also Critical

Embedding 0 must always correspond to the same variable.

For example:

```text
embedding[0] -> sex
embedding[1] -> symptom_group
```

Do not reorder columns after training.


# 109. Class Mapping Must Also Be Saved

For example:

```python
{
    0: "Class A",
    1: "Class B",
    2: "Class C"
}
```

The model checkpoint alone is not enough.


# 110. Multimodal Error Analysis

For every error, inspect:

- Ultrasound image
- Clinical feature vector
- Missing-value indicators
- Model confidence
- Image-only prediction
- Tabular-only prediction
- Fused prediction

This can reveal conflicts between modalities.


# 111. Modality Conflict

Example:

$$
Image\ Model
\rightarrow
Class\ 0
$$

$$
Clinical\ Model
\rightarrow
Class\ 2
$$

$$
Fusion\ Model
\rightarrow
Class\ 2
$$

This tells us the fusion model trusted the clinical branch more.

Such cases are valuable for debugging.


# 112. Agreement and Disagreement Analysis

Create groups:

- Both modalities correct
- Image correct / clinical wrong
- Image wrong / clinical correct
- Both wrong

Then study when fusion helps.


# 113. Fusion Gain

Define:

$$
FusionGain
=
Metric_{fusion}
-
Metric_{best\ single\ modality}
$$

A multimodal method should ideally show positive gain across:

- Folds
- Seeds
- Sites


# 114. Why Fusion Gain Can Be Small

If image and clinical features contain mostly the same information, fusion may add little.

Multimodal learning helps most when modalities provide complementary information.


# 115. Why Tabular-Only Performance Can Be Suspiciously High

If a clinical-only model dramatically outperforms imaging:

Investigate:

- Leakage
- Outcome-coded variables
- Post-diagnosis variables
- Missingness shortcuts
- Site/label confounding

It may be valid—but deserves careful inspection.


# 116. Why Image-Only Performance Can Dominate

Possible reasons:

- Clinical variables are noisy
- Clinical features have too much missingness
- Clinical variables are weak predictors
- Tabular preprocessing is poor
- Dataset is too small for multimodal learning


# 117. Fusion Model Calibration

Even if fusion improves accuracy/AUROC, calibration may change.

If probabilities are used clinically, compare:

- Image-only calibration
- Tabular-only calibration
- Fusion calibration


# 118. Threshold Selection in Multimodal Binary Classification

For a binary task:

- Train multimodal model
- Generate validation probabilities
- Select threshold on validation only
- Freeze threshold
- Evaluate test once

Do not choose a different threshold for each test site after seeing outcomes.


# 119. Explainability in Multimodal Models

Image explanation methods such as Grad-CAM only explain the image branch.

They do not explain the entire fused decision.

You also need to inspect:

- Clinical feature influence
- Modality contribution
- Image/tabular disagreement


# 120. Tabular Feature Importance Caution

Feature importance methods can be unstable and correlated features complicate interpretation.

Use feature importance as:

> **A debugging/analysis tool**

not proof of causality.


# 121. Counterfactual Clinical Debugging

One useful question:

> What happens to prediction if age changes slightly while everything else remains fixed?

This can reveal surprising model sensitivity.

But such counterfactuals may be clinically unrealistic if variables are correlated.


# 122. Modality-Specific Explainability

Possible tools:

## Image

- Grad-CAM
- Saliency
- Occlusion

## Tabular

- Feature ablation
- Permutation
- Gradient sensitivity
- SHAP-style methods

## Fusion

- Remove one modality
- Compare logits
- Gate/attention weights


# 123. Modality Ablation Is Often the Most Reliable First Explanation

Ask:

$$
Prediction_{both}
$$

then:

$$
Prediction_{image\ only}
$$

then:

$$
Prediction_{tabular\ only}
$$

This directly shows how prediction changes when one modality is removed.


# 124. Research-Quality Multimodal Protocol

A strong study might use:

1. Patient-grouped folds
2. Training-only preprocessing
3. Image-only baseline
4. Tabular-only baseline
5. Fusion model
6. Same folds/seeds for all models
7. Site/device subgroup analysis
8. Missing-modality evaluation
9. External validation
10. Patient-level confidence intervals


# 125. Multimodal Result Table

A useful result table might contain:

$$
\begin{array}{|c|c|c|c|c|}
\hline
Model &
Internal &
External &
Worst\ Site &
Missing\ Modality \\
\hline
Image\ Only & \cdots & \cdots & \cdots & N/A \\
\hline
Clinical\ Only & \cdots & \cdots & \cdots & N/A \\
\hline
Fusion & \cdots & \cdots & \cdots & \cdots \\
\hline
\end{array}
$$


# 126. Common Mistake — Clinical Preprocessing Before Splitting

Wrong:

```text
all patients
→ compute medians/mean/std
→ split
```

Correct:

```text
split patients
→ fit preprocessing on train
→ apply to validation/test
```


# 127. Common Mistake — Recomputing Categories on Test Data

Do not rebuild categorical vocabularies on the test set.

Map unseen categories to:

```text
__UNK__
```


# 128. Common Mistake — Outcome Leakage

Never include variables generated after the clinical endpoint you are trying to predict.


# 129. Common Mistake — Mixing Patient-Level and Image-Level Splits

All modalities for one patient must belong to the same split.

Do not split images and tabular rows independently.


# 130. Common Mistake — Comparing Different Folds

Image-only and fused models must use identical patient folds for fair comparison.


# 131. Common Mistake — Ignoring Missingness

Replacing missing values without indicators may hide clinically meaningful missingness patterns.

But missingness can also be a workflow shortcut.

Analyze both possibilities.


# 132. Common Mistake — Feeding Site as a Predictor Without Justification

Site can become a direct shortcut for disease prevalence.

Use it for evaluation/harmonization unless there is a strong reason for including it in prediction.


# 133. Common Mistake — Very Large Tabular Encoder

A large MLP can overfit a small clinical dataset quickly.

Keep the branch compact.


# 134. Common Mistake — Huge Fusion Head

The fusion head should not dwarf the available patient sample size.


# 135. Common Mistake — Assuming Fusion Helps Because Validation Improved Once

Repeat:

- Multiple folds
- Multiple seeds
- External validation

before claiming multimodal benefit.


# 136. Common Mistake — Ignoring Clinical Unit Differences Across Sites

Neural-network scaling does not replace proper unit conversion.


# 137. Common Mistake — Treating Missing Modality as a Zero Vector Without a Mask

The model cannot distinguish:

> real zero features

from:

> missing modality

without an explicit indicator.


# 138. Common Mistake — Evaluating Only With Both Modalities Present

Real deployment may have missing clinical data.

Test realistic missing-modality scenarios.


# 139. Practice Exercises

## Exercise 1

Create numerical and categorical clinical features for a patient table.

## Exercise 2

Fit numerical medians using training patients only.

## Exercise 3

Standardize numerical variables using training mean/std.

## Exercise 4

Create categorical vocabularies with `__UNK__` and `__MISSING__`.

## Exercise 5

Build an image encoder that returns a 64-dimensional vector.

## Exercise 6

Build a tabular encoder using embeddings.

## Exercise 7

Concatenate image and clinical features and build a fusion classifier.

## Exercise 8

Compare image-only, tabular-only, and fused models.

## Exercise 9

Aggregate image predictions to patient level.

## Exercise 10

Add modality-presence masks for missing-modality inference.


# 140. Conceptual Challenges

## Challenge 1

Why can clinical features improve an ultrasound model?

## Challenge 2

Why can adding clinical variables hurt performance?

## Challenge 3

What is prediction-time leakage?

## Challenge 4

Why should numerical preprocessing be fitted on training patients only?

## Challenge 5

Why add missingness indicators?

## Challenge 6

Why use an `__UNK__` category?

## Challenge 7

What is the difference between early, intermediate, and late fusion?

## Challenge 8

Why must image-only and fused models use identical patient folds?

## Challenge 9

Why can repeated clinical variables overweight patients with many images?

## Challenge 10

Why should missing-modality performance be evaluated?

## Challenge 11

Why can site/device metadata become shortcuts?

## Challenge 12

Why is patient-level evaluation important?

## Challenge 13

Why can tabular-only performance reveal leakage?

## Challenge 14

Why is multimodal external validation especially important?

## Challenge 15

What evidence should support the claim that fusion truly adds value?


# 141. Exercise Solutions


In [ ]:
# Exercise 2 and 3
exercise_numeric_columns = [
    "age",
    "bmi"
]

exercise_preprocessor = (
    ClinicalPreprocessor(
        exercise_numeric_columns,
        []
    )
    .fit(
        train_df
    )
)

print(
    exercise_preprocessor
    .numeric_medians
)

print(
    exercise_preprocessor
    .numeric_means
)

print(
    exercise_preprocessor
    .numeric_stds
)


In [ ]:
# Exercise 4
exercise_category_preprocessor = (
    ClinicalPreprocessor(
        [],
        [
            "sex",
            "symptom_group"
        ]
    )
    .fit(
        train_df
    )
)

print(
    exercise_category_preprocessor
    .category_maps
)


In [ ]:
# Exercise 5
exercise_image_encoder = (
    ImageEncoder(
        output_dim=64
    )
)

exercise_image_features = (
    exercise_image_encoder(
        batch_images
    )
)

print(
    exercise_image_features.shape
)


In [ ]:
# Exercise 6
exercise_tabular_encoder = (
    TabularEncoder(
        numeric_dim=numeric_dim,
        category_cardinalities=(
            category_cardinalities
        ),
        embedding_dim=4,
        output_dim=32
    )
)

exercise_tab_features = (
    exercise_tabular_encoder(
        batch_numeric,
        batch_categorical
    )
)

print(
    exercise_tab_features.shape
)


In [ ]:
# Exercise 7
exercise_fused = torch.cat(
    [
        exercise_image_features,
        exercise_tab_features
    ],
    dim=1
)

exercise_head = nn.Linear(
    exercise_fused.shape[
        1
    ],
    3
)

exercise_logits = exercise_head(
    exercise_fused
)

print(
    exercise_logits.shape
)


In [ ]:
# Exercise 9
exercise_predictions = (
    collect_predictions(
        fusion_model,
        val_dataset,
        val_loader,
        mode="fusion",
        device=device
    )
)

exercise_patient_predictions = (
    aggregate_patient_predictions(
        exercise_predictions,
        num_classes=3
    )
)

print(
    exercise_patient_predictions.head()
)


In [ ]:
# Exercise 10
exercise_image_mask, exercise_tab_mask = (
    sample_modality_masks(
        batch_size=(
            batch_images.shape[
                0
            ]
        ),
        image_drop_probability=0.2,
        tabular_drop_probability=0.2
    )
)

print(
    exercise_image_mask.shape,
    exercise_tab_mask.shape
)


# 142. Conceptual Challenge Solutions

## Challenge 1

Clinical variables can provide complementary risk or physiological context not visible in the image.

## Challenge 2

They may add noise, missingness, site-specific shortcuts, or outcome leakage, and they increase model capacity.

## Challenge 3

Prediction-time leakage occurs when a feature would not actually be available when the intended prediction must be made.

## Challenge 4

Using validation/test patients to calculate medians, means, or standard deviations leaks information into preprocessing.

## Challenge 5

After imputation, the model otherwise cannot distinguish a truly observed median value from a missing value replaced by the median.

## Challenge 6

External data may contain categories never seen in training. `__UNK__` provides a stable mapping without refitting.

## Challenge 7

Early fusion combines modalities near the input, intermediate fusion combines learned features, and late fusion combines modality-level predictions.

## Challenge 8

Otherwise differences may reflect patient difficulty rather than the fusion method.

## Challenge 9

If one patient's clinical vector is repeated for many images, that patient contributes many more training examples.

## Challenge 10

Real deployment may not always provide every modality. A robust model should degrade predictably.

## Challenge 11

Site/device can correlate with labels and become easier shortcuts than clinically meaningful signal.

## Challenge 12

If the clinical decision is per patient, each patient should ultimately contribute one evaluation decision.

## Challenge 13

Suspiciously high tabular performance can indicate outcome-coded variables, workflow leakage, or missingness shortcuts.

## Challenge 14

Both image and clinical distributions can shift across hospitals, so internal fusion gains may not generalize.

## Challenge 15

Fusion should outperform well-tuned single-modality baselines across predefined folds/seeds, preserve subgroup robustness, and ideally show benefit on independent external patients.


# 143. Key Takeaways

In this notebook, we studied:

- Why multimodal learning can help
- Why more modalities can also hurt
- Prediction-time leakage
- Patient-level multimodal splitting
- Numerical clinical features
- Categorical clinical features
- Missing numerical values
- Missingness indicators
- Unknown categories
- Training-only clinical preprocessing
- Training-only image normalization
- Image encoders
- Tabular encoders
- Embeddings
- Feature fusion
- Early fusion
- Intermediate fusion
- Late fusion
- Image-only baselines
- Tabular-only baselines
- Concatenation-based fusion
- Multimodal training loops
- Modality ablation studies
- Patient-level aggregation
- Missing modalities
- Modality-presence masks
- Modality dropout
- Site/device-aware multimodal evaluation
- External multimodal validation
- Reproducible preprocessing configs
- Multimodal error analysis

The core architecture is:

$$
\boxed{
Image
\rightarrow
ImageEncoder
\rightarrow
z_{image}
}
$$

$$
\boxed{
Clinical
\rightarrow
TabularEncoder
\rightarrow
z_{clinical}
}
$$

$$
\boxed{
[z_{image};z_{clinical}]
\rightarrow
FusionHead
\rightarrow
Prediction
}
$$

The most important leakage rule is:

$$
\boxed{
Only\ Features\ Available\ at\ Prediction\ Time
}
$$

The most important evaluation rule is:

$$
\boxed{
ImageOnly
\ vs
ClinicalOnly
\ vs
Fusion
}
$$

under the same patient folds and seeds.


# 144. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What is a modality?
2. Why combine ultrasound with clinical features?
3. Why can multimodal learning hurt?
4. What is prediction-time leakage?
5. Why should patient splitting happen before clinical preprocessing?
6. Why use training medians for missing numerical values?
7. Why standardize numerical variables?
8. What is a missingness indicator?
9. Why use categorical embeddings?
10. What is `__UNK__`?
11. What is `__MISSING__`?
12. What does the image encoder output?
13. What does the tabular encoder output?
14. What is feature fusion?
15. What is early fusion?
16. What is intermediate fusion?
17. What is late fusion?
18. Why train image-only and tabular-only baselines?
19. What is a modality ablation?
20. Why can fusion improvement indicate leakage?
21. Why can repeated clinical features overweight patients?
22. Why should multimodal evaluation be patient-level when the target is patient-level?
23. What is a missing modality?
24. What is a modality-presence mask?
25. What is modality dropout?
26. Why should both modalities not usually be absent simultaneously?
27. Why can site/device metadata be dangerous predictor inputs?
28. Why must clinical vocabularies stay fixed for external validation?
29. Why can missingness patterns shift across sites?
30. What evidence should show that multimodal fusion truly improves a research-quality ultrasound model?


# Next Notebook

# 28 — Multi-View, Multi-Frame, and Study-Level Ultrasound Learning

In the next notebook, we will study:

- Why one image may not represent a full ultrasound study
- Multiple images per patient
- Multiple views per examination
- Cine-loop and frame sequences
- Image-level vs study-level modeling
- Mean and max probability aggregation
- Majority voting
- Learned feature aggregation
- Attention-based pooling
- Multi-instance learning intuition
- Patient-balanced sampling
- Variable numbers of images per study
- Masked batching for study-level models
- Preventing leakage across frames and studies
- Study-level evaluation
- Preparing a multi-view ultrasound research pipeline
